# Lỗi còn lại sau chuẩn hóa — danh sách để sửa tay

Notebook này **chỉ đọc** `data/crawl_v2_clean.json` (đầu ra của `normalize.py`) và liệt kê những chỗ mà quy tắc không tự sửa được. Mỗi chỗ có: link trang gốc, file đính kèm (link Drive), nơi lưu trong dữ liệu, số dòng và đoạn chữ lỗi.

- Số dòng tính **trong trường** của bản ghi (ví dụ `extracted_text` của `tb-2234`), không phải số dòng trong file PDF.
- Sửa tay trong `crawl_v2_clean.json` sẽ **bị ghi đè** nếu chạy lại `python normalize.py run`.
- Danh sách đầy đủ được ghi ra `extract/loi_can_sua.csv`.

In [1]:
import html, json, re, sys
from collections import Counter
from pathlib import Path
from datetime import date

import pandas as pd
from IPython.display import HTML, display

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
import fix_ocr as F

records = json.loads((ROOT / "data" / "crawl_v2_clean.json").read_text(encoding="utf-8"))
vocab = F.build_vocab()[0]          # từ vựng lấy từ chữ KHÔNG qua OCR
ocr_ids = [r["id"] for r in records if "ocr" in (r.get("extract_method") or "")]
print(f"{len(records)} bản ghi, trong đó {len(ocr_ids)} bản ghi có chữ từ OCR: {ocr_ids}")

141 bản ghi, trong đó 7 bản ghi có chữ từ OCR: ['tb-2234', 'tb-2235', 'tb-2254', 'tb-2223', 'tb-2248', 'tb-2284', 'tb-2281']


## 1. Các kiểm tra

| Loại lỗi | Áp dụng cho | Cách phát hiện |
|---|---|---|
| Ngày không hợp lệ / chưa đệm 0 | mọi bản ghi | `d/m/yyyy` mà ngày không tồn tại hoặc không phải `dd/mm/yyyy` |
| Mã môn còn `I/l/O` ở phần số | bản ghi OCR | 3 chữ + 4 ký tự `[0-9IlO]` có chứa `I/l/O` (mã đúng đã tự sửa, đây là phần không có mã đúng để đối chiếu) |
| Số lẫn chữ `I/l/O` | bản ghi OCR | token riêng 2–3 ký tự gồm số và `I/l/O` |
| Từ có dấu không có trong từ vựng sạch | bản ghi OCR | từ ≥ 8 chữ cái, có dấu tiếng Việt, không xuất hiện trong chữ không qua OCR |
| Dòng chữ rời | bản ghi OCR | dòng gồm ≥ 6 ký tự đơn cách nhau bằng dấu cách (`X X X X`) |

In [2]:
FIELDS = ("title", "content", "extracted_text")
DATE_ANY = re.compile(r"(?<![\d/.])(\d{1,2})/(\d{1,2})/(\d{4})(?!\d)")
LONE_LETTERS = re.compile(r"(?m)^(?:\S[ \t]){5,}\S$")
WORD = re.compile(r"[^\W\d_]+")


def date_problem(m):
    d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
    try:
        date(y, mo, d)
    except ValueError:
        return "ngày không hợp lệ (nghi OCR mất/đọc sai chữ số)"
    return "ngày chưa đệm 0" if m.group(0) != f"{d:02d}/{mo:02d}/{y}" else None


def find_todo():
    todo = []
    for r in records:
        is_ocr = r["id"] in ocr_ids
        atts = {a["name"]: a for a in (r.get("attachments") or [])}
        for f in FIELDS:
            t = r.get(f) or ""
            if not t:
                continue
            heads = [(i, ln[1:-1]) for i, ln in enumerate(t.split("\n")) if ln.startswith("[") and ln.endswith("]")]

            def add(kind, m):
                line = t.count("\n", 0, m.start())
                name = None
                for i, h in heads:                    # file đính kèm = tiêu đề [tên file] gần nhất phía trên
                    if i <= line and h in atts:
                        name = h
                a = atts.get(name, {})
                todo.append({
                    "record_id": r["id"], "link_trang": r.get("source_url", ""),
                    "file_dinh_kem": name or "", "link_file": a.get("url", ""), "file_key": a.get("file_key", ""),
                    "dong_day_du": t[t.rfind("\n", 0, m.start()) + 1:(t.find("\n", m.end()) if t.find("\n", m.end()) >= 0 else len(t))],
                    "truong": f, "dong": line + 1, "loai_loi": kind, "noi_dung_loi": m.group(0),
                    "ngu_canh": t[max(0, m.start() - 40):m.end() + 40].replace("\n", " ⏎ "),
                    "hl_start": m.start() - max(0, m.start() - 40), "hl_end": m.end() - max(0, m.start() - 40)})

            for m in DATE_ANY.finditer(t):
                kind = date_problem(m)
                if kind:
                    add(kind, m)
            if not (is_ocr and f == "extracted_text"):
                continue
            for m in F.CODE_TOKEN.finditer(t):
                if re.search(r"[IlO]", m.group(2)):
                    add("mã môn còn chữ I/l/O ở phần số", m)
            for m in F.DIGIT_TOKEN.finditer(t):
                add("số lẫn chữ I/l/O", m)
            for m in WORD.finditer(t):
                w = m.group(0).lower()
                if len(w) >= 8 and F.VN_LETTER.search(w) and w not in vocab:
                    add("từ có dấu không có trong từ vựng sạch", m)
            for m in LONE_LETTERS.finditer(t):
                add("dòng chữ rời (nhiễu bảng)", m)
    return todo


todo = find_todo()
df = pd.DataFrame(todo)
print(f"{len(df)} chỗ cần xem, {df['record_id'].nunique()} bản ghi")

529 chỗ cần xem, 3 bản ghi


## 2. Tổng hợp số lỗi

In [3]:
summary = df.pivot_table(index="loai_loi", columns="record_id", values="dong", aggfunc="count", fill_value=0)
summary["Tổng"] = summary.sum(axis=1)
summary.loc["Tổng"] = summary.sum()
summary

record_id,tb-2234,tb-2235,tb-2284,Tổng
loai_loi,,,,
dòng chữ rời (nhiễu bảng),37,2,0,39
mã môn còn chữ I/l/O ở phần số,91,1,0,92
ngày không hợp lệ (nghi OCR mất/đọc sai chữ số),0,0,1,1
số lẫn chữ I/l/O,0,2,0,2
từ có dấu không có trong từ vựng sạch,248,146,1,395
Tổng,376,151,2,529


## 3. Xem từng chỗ lỗi

Với mỗi (bản ghi, loại lỗi): link trang, file đính kèm, nơi sửa, rồi 15 chỗ đầu (đầy đủ trong CSV). Phần lỗi được tô vàng.

In [4]:
SHOW = 15


def link(url, text):
    return f'<a href="{html.escape(url)}" target="_blank">{html.escape(text)}</a>' if url else html.escape(text)


parts = []
for (rid, kind), g in df.groupby(["record_id", "loai_loi"], sort=True):
    x0 = g.iloc[0]
    atts = g[["file_dinh_kem", "link_file"]].drop_duplicates()
    att_html = "; ".join(link(a.link_file, a.file_dinh_kem or "(không rõ)") for a in atts.itertuples())
    rows = []
    for x in g.head(SHOW).itertuples():
        c = x.ngu_canh
        snip = html.escape(c[:x.hl_start]) + "<mark>" + html.escape(c[x.hl_start:x.hl_end]) + "</mark>" + html.escape(c[x.hl_end:])
        rows.append(f"<tr><td>{x.dong}</td><td style='font-family:monospace;white-space:pre-wrap'>…{snip}…</td></tr>")
    more = f"<p>… còn {len(g) - SHOW} chỗ nữa (xem CSV).</p>" if len(g) > SHOW else ""
    parts.append(
        f"<h4>[{html.escape(rid)}] {html.escape(kind)}: {len(g)} chỗ</h4>"
        f"<p>Trang: {link(x0.link_trang, x0.link_trang)} &nbsp;|&nbsp; File đính kèm: {att_html}<br>"
        f"Sửa trong: <code>data/crawl_v2_clean.json &gt; {html.escape(rid)} &gt; {x0.truong}</code></p>"
        f"<table><tr><th>Dòng</th><th>Nội dung</th></tr>{''.join(rows)}</table>{more}")
display(HTML("".join(parts)))

Dòng,Nội dung
995,"…g PI ⏎ ⏎ (LO) ⏎ ⏎ PI ⏎ (Performance Indicator) ⏎ ⏎ H À Ầ H 2 "" . ⏎ Chuẩn đâu ra về mức tự chủ và chịu trác…"
19169,"…am | Ễ mW IR ⏎ 57 Êi chính | = M. R, M.ỊR ⏎ ` = A A A | A ⏎ nâng cao | - ⏎ mm HE BI 1. IaE ⏎ 58 ng 2 R …"
35599,"… ⏎ | "") | Học phân băt buộc .23 | | ⏎ 1 J l ⏎ | | | | | | Ỉ | | ⏎ l Triết học Mác - Lênin PLE1101 3145 | …"
35606,…h | PLE1111 2. 30 30 PLE1102 | | x Lo I ⏎ l | J l | | | | ⏎ TT; DAI ĩ ĩ Ĩ ⏎ | 5 Eichsở Đăng Cộng PLE1…
35646,… 260 | F := ⏎ và pháp luật | | | s.. -~za ⏎ n 1 Ĩ Ị | | ⏎ 15 CáchocthuyetebfS | qwiai2] $ | d5 45…
35691,…| | ⏎ Luật hình sự LAWI1422| 3 45 45 .x ⏎ i ⏎ P | | ! ! | | | \ | ⏎ Luật tô tụng hình sự LAWI1432 2 15 30 7…
35724,…ở hữu trí tuệ LAW1464 3 45 45 li | xX | ⏎ | ầ H Ồ Ị | ⏎ Ô..-40 in thương mại QUỐC |1 Awr473| 3 …
35755,…0 | 30.60 | LAWI1452 ] Lx| ⏎ đai | | | -_ ⏎ T | | | | ! ⏎ 47 Luật đầu tư LAW1506 3. 45 45 | | mrs…
35770,…1.30 |LAW1443 | mm N ⏎ động sản | | Ị J | ⏎ Z = ! | | | | | | ⏎ H3 Kiến thức bỗ trợ | 6 | | | | mưum ⏎ Ng…
35775,… 1.4 | | | | ⏎ ngoại ngữ sau) | | | _ L.) ⏎ P | | | | | ⏎ 55 Tiếng Pháp 1 FRAU L4 |] mg j 130 | 4…


## 4. Ghi danh sách đầy đủ ra CSV

In [5]:
out = ROOT / "extract" / "loi_can_sua.csv"
df.drop(columns=["hl_start", "hl_end"]).assign(file_du_lieu=lambda d: "data/crawl_v2_clean.json > " + d["record_id"] + " > " + d["truong"]) \
  .to_csv(out, index=False, encoding="utf-8-sig")
print("Đã ghi", len(df), "dòng ->", out)

Đã ghi 529 dòng -> C:\A47752_HaDuyAnh\TTTN\standardization\extract\loi_can_sua.csv


## 5. Xác định số trang PDF của từng lỗi

Chữ OCR được lưu theo từng trang (`extract/ocr/<file>/pNNNN.json`). Với mỗi lỗi, chương trình tìm chữ sai trong từng trang của **đúng file đính kèm**:

- chỉ có 1 trang chứa chữ sai → số trang đó (*chắc chắn*);
- nhiều trang chứa → thu hẹp bằng đoạn chữ xung quanh trong cùng dòng; nếu vẫn nhiều trang thì ghi **tất cả các trang ứng viên**;
- không thấy (chữ đã bị chuẩn hóa khác chữ OCR gốc) → ghi **không xác định**.

Số trang là số thứ tự trang trong file PDF (trang 1 = trang đầu), khớp với số trang của trình xem PDF.

In [6]:
OCR_DIR = ROOT / "extract" / "ocr"
_pages = {}


def squash(s):
    return re.sub(r"\s+", " ", s)


def load_pages(key):
    if key not in _pages:
        out = []
        for p in sorted((OCR_DIR / key).glob("p*.json")):
            j = json.loads(p.read_text(encoding="utf-8"))
            out.append((j["page"], squash(j["text"])))
        _pages[key] = out
    return _pages[key]


def locate(row):
    """(trang, mức): mức = 'chắc chắn' | 'nhiều trang' | 'không xác định'."""
    if not row.file_key:
        return "", "không xác định"
    pages = load_pages(row.file_key)
    tok = squash(row.noi_dung_loi)
    hit = [pg for pg, tx in pages if tok in tx]
    if len(hit) == 1:
        return str(hit[0]), "chắc chắn"
    if len(hit) > 1:
        line = squash(row.dong_day_du)
        i = line.find(tok)
        for w in (40, 25, 15):                        # cửa sổ chữ quanh chỗ lỗi, thu hẹp dần
            win = line[max(0, i - w):i + len(tok) + w] if i >= 0 else ""
            sub = [pg for pg in hit if win and win in dict(pages)[pg]]
            if len(sub) == 1:
                return str(sub[0]), "chắc chắn"
        return ", ".join(str(p) for p in hit[:6]) + ("…" if len(hit) > 6 else ""), "nhiều trang"
    return "", "không xác định"


res = [locate(r) for r in df.itertuples()]
df["trang"] = [r[0] for r in res]
df["muc_trang"] = [r[1] for r in res]
print(df["muc_trang"].value_counts().to_string())
print()
print(df.groupby(["record_id", "muc_trang"]).size().unstack(fill_value=0))

muc_trang
chắc chắn         481
không xác định     37
nhiều trang        11

muc_trang  chắc chắn  không xác định  nhiều trang
record_id                                        
tb-2234          338              30            8
tb-2235          141               7            3
tb-2284            2               0            0


## 6. Xuất Excel để điền chữ đúng

File `data/loi_can_sua.xlsx` có 2 sheet:

- **Hướng dẫn**: cách làm từng bước, cách điền cho từng nhóm lỗi, ví dụ.
- **Lỗi OCR**: mỗi dòng một chỗ lỗi, sắp theo bản ghi, file, số trang, dòng, để làm lần lượt từ đầu đến cuối mỗi file PDF.

Nhóm lỗi chỉ đổi tên từ loại lỗi phát hiện ở mục 1; không có phân loại "tên học phần" hay "tên riêng" vì không có căn cứ tự động để phân biệt.

In [7]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation

NHOM = {
    "mã môn còn chữ I/l/O ở phần số": "Sai mã học phần",
    "ngày không hợp lệ (nghi OCR mất/đọc sai chữ số)": "Sai ngày tháng",
    "ngày chưa đệm 0": "Sai ngày tháng",
    "số lẫn chữ I/l/O": "Sai số (lẫn chữ)",
    "từ có dấu không có trong từ vựng sạch": "Sai chữ (từ dính, sai dấu)",
    "dòng chữ rời (nhiễu bảng)": "Nhiễu bảng (dòng ký tự rời)",
}
HEAD = ["STT", "Lỗi mắc phải (ví dụ như sai mã học phần, tên, ...)", "Chữ sai", "Dòng chứa lỗi (để dễ tìm)",
        "File gốc", "Trang trong file", "Link gốc", "Khắc phục (điền chữ đúng vào đây)", "Ghi chú (nếu cần)",
        "Vị trí trong dữ liệu (để máy áp dụng, đừng sửa)"]
THIN = Side(style="thin", color="BFBFBF")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

wb = Workbook()

# ---- sheet hướng dẫn
g = wb.active
g.title = "Hướng dẫn"
g.column_dimensions["A"].width = 34
g.column_dimensions["B"].width = 95
rows = [
    ("HƯỚNG DẪN SỬA LỖI DỮ LIỆU", None),
    ("Mục đích", "Văn bản trong các file PDF dạng ảnh được máy đọc chữ (OCR) nên có chỗ đọc sai. Danh sách ở sheet “Lỗi OCR” là những chỗ máy phát hiện nghi sai. Bạn đối chiếu với file PDF gốc và điền chữ đúng vào cột “Khắc phục”."),
    ("", ""),
    ("CÁCH LÀM TỪNG BƯỚC", None),
    ("Bước 1", "Mở sheet “Lỗi OCR”. Các dòng đã được sắp theo file và theo trang, nên làm lần lượt từ trên xuống."),
    ("Bước 2", "Bấm vào link ở cột “Link gốc” để mở file PDF (trên Google Drive)."),
    ("Bước 3", "Mở đến trang ghi ở cột “Trang trong file”. Nếu ghi nhiều trang (ví dụ 12, 13) thì xem lần lượt các trang đó. Nếu để trống thì dùng cột “Dòng chứa lỗi” để tìm bằng mắt (ví dụ tìm tên môn học)."),
    ("Bước 4", "Tìm đúng chỗ ở cột “Dòng chứa lỗi”. Phần chữ sai là nội dung ở cột “Chữ sai” (máy đọc từ ảnh nên có thể lẫn ký tự lạ)."),
    ("Bước 5", "Điền chữ ĐÚNG, giống như in trên PDF, vào cột “Khắc phục”. Chỉ điền phần chữ sai, không điền cả dòng."),
    ("", ""),
    ("CÁCH ĐIỀN THEO NHÓM LỖI", None),
    ("Sai mã học phần", "Điền mã đúng. Ví dụ: Chữ sai “NWSI403” → PDF in “NWS1403” → điền: NWS1403. (Mã học phần gồm 3 chữ cái và 4 chữ số; máy hay đọc số 1 thành chữ I hoặc l, số 0 thành chữ O.)"),
    ("Sai ngày tháng", "Điền ngày đúng theo dạng dd/mm/yyyy. Ví dụ: “05/0/2027” → PDF in 05/07/2027 → điền: 05/07/2027."),
    ("Sai số (lẫn chữ)", "Điền số đúng. Nếu thực ra đó là ký hiệu hợp lệ (không phải số) thì điền: [GIỮ]."),
    ("Sai chữ (từ dính, sai dấu)", "Điền cụm chữ đúng, có dấu cách và dấu tiếng Việt. Ví dụ: “Tinhọocứngdụngtrong” → điền: Tin học ứng dụng trong."),
    ("Nhiễu bảng (dòng ký tự rời)", "Thường là rác do đường kẻ bảng. Nếu trong PDF không có chữ nào như vậy thì điền: [XÓA]. Nếu đó là nội dung thật (ví dụ dấu X trong bảng đối chiếu) thì điền: [GIỮ]."),
    ("Chuỗi rác không phải chữ", "Với mọi nhóm: nếu chỗ đó chỉ là ký tự rác, không có nội dung thật, điền: [XÓA]."),
    ("", ""),
    ("Ý NGHĨA CÁC CỘT", None),
    ("Trang trong file", "Số trang của file PDF (trang 1 là trang đầu), do máy tự tìm. Trống = không tìm được. Nhiều số = chữ sai xuất hiện ở nhiều trang."),
    ("Dòng chứa lỗi", "Nguyên dòng chữ chứa lỗi, theo chữ máy đọc được (đã làm sạch dấu cách thừa)."),
    ("Vị trí trong dữ liệu", "Mã bản ghi · trường · số dòng trong file crawl_v2_clean.json. Cột này để chương trình áp dụng sửa, xin đừng chỉnh."),
    ("", ""),
    ("LƯU Ý", None),
    ("Mẫu không đại diện", "Đây chỉ là những chỗ máy nghi sai, không phải mọi lỗi của văn bản. Danh sách không dùng để suy ra tỷ lệ lỗi chung."),
    ("Chưa chắc là lỗi", "Một số dòng máy nghi sai nhưng chữ lại đúng (tên riêng, từ nước ngoài). Khi đó điền: [GIỮ]."),
]
for a, b in rows:
    g.append([a, b])
    r = g.max_row
    if b is None and a:
        g.cell(r, 1).font = Font(bold=True, size=12, color="FFFFFF")
        g.cell(r, 1).fill = PatternFill("solid", fgColor="305496")
        g.cell(r, 2).fill = PatternFill("solid", fgColor="305496")
    else:
        g.cell(r, 1).font = Font(bold=True)
    g.cell(r, 1).alignment = Alignment(vertical="top", wrap_text=True)
    g.cell(r, 2).alignment = Alignment(vertical="top", wrap_text=True)
g["A1"].font = Font(bold=True, size=14, color="FFFFFF")

# ---- sheet lỗi
ws = wb.create_sheet("Lỗi OCR")
ws.append(HEAD)
order = df.assign(_p=pd.to_numeric(df["trang"].str.extract(r"(\d+)")[0], errors="coerce").fillna(10**6)) \
          .sort_values(["record_id", "file_dinh_kem", "_p", "dong"], kind="stable")
for n, x in enumerate(order.itertuples(), 1):
    url = x.link_file or x.link_trang
    ws.append([
        n, NHOM[x.loai_loi], x.noi_dung_loi, x.dong_day_du[:300], x.file_dinh_kem or "(nội dung trang web)",
        x.trang, url, None, None, f"{x.record_id} · {x.truong} · dòng {x.dong}"])
    c = ws.cell(row=ws.max_row, column=7)
    c.hyperlink = url
    c.font = Font(color="0563C1", underline="single")
    ws.cell(row=ws.max_row, column=8).fill = PatternFill("solid", fgColor="FFF2CC")

for cell in ws[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(wrap_text=True, vertical="center")
for row in ws.iter_rows(min_row=2):
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical="top")
        c.border = BORDER
    row[9].font = Font(color="808080", size=9)
for col, w in zip("ABCDEFGHIJ", (6, 24, 24, 60, 32, 12, 42, 34, 24, 26)):
    ws.column_dimensions[col].width = w
ws.row_dimensions[1].height = 34
ws.freeze_panes = "D2"
ws.auto_filter.ref = ws.dimensions
dv = DataValidation(type="custom", formula1="TRUE", allow_blank=True, showInputMessage=True,
                    promptTitle="Khắc phục", prompt="Điền chữ ĐÚNG như in trên PDF. Rác: [XÓA]. Chữ đã đúng: [GIỮ].")
ws.add_data_validation(dv)
dv.add(f"H2:H{ws.max_row}")

xlsx = ROOT / "data" / "loi_can_sua.xlsx"
wb.save(xlsx)
print("Đã ghi", ws.max_row - 1, "dòng ->", xlsx)

Đã ghi 529 dòng -> C:\A47752_HaDuyAnh\TTTN\standardization\data\loi_can_sua.xlsx
